In [10]:
from pathlib import Path

import numpy as np
import pandas as pd


from pathlib import Path

# Law el notebook shaghala mn gowwa folder 'src/data':
ROOT = Path.cwd().parents[1]  # Byerga3 2 levels le fo2 le 'City-Hotel-Booking'

# Check eno wasal le root el project sa7:
print("Root:", ROOT)

RAW_PATH = ROOT / "data" / "H2.csv"
CLEAN_PATH = ROOT / "data" / "processed" / "H2_clean.csv"
REPORT_PATH = ROOT / "outputs" / "cleaning_report.csv"

# Values outside this are invalid data (the 5400 typo / negatives), not just outliers.
# ADR == 0 is NOT handled here (task-specific).
ADR_MAX_VALID = 1000

MONTHS = {m: i for i, m in enumerate(
    ["January", "February", "March", "April", "May", "June", "July",
     "August", "September", "October", "November", "December"], start=1)}



def _id_to_str(s):
    """Agent/Company ids -> clean string ids; missing -> 'None'."""
    s = pd.to_numeric(s, errors="coerce")
    # If value_counts() shows 0 is the 'missing' code in your mirror, uncomment:
    # s = s.mask(s == 0)
    return s.map(lambda v: "None" if pd.isna(v) else str(int(v)))

# Global audit tracking log
log = []

def track_drop(df_current, mask, reason):
    dropped_count = int(mask.sum())
    log.append((reason, dropped_count))
    print(f"[*] {reason:<22}: Dropped {dropped_count:>5} rows | Remaining: {len(df_current) - dropped_count}")
    return df_current.loc[~mask]

print("Root path resolved:", ROOT)
print("RAW_PATH exists?", RAW_PATH.exists())

Root: d:\UNI\Fall 2026\CSE382- Intro. to ML\Project\City-Hotel-Booking
Root path resolved: d:\UNI\Fall 2026\CSE382- Intro. to ML\Project\City-Hotel-Booking
RAW_PATH exists? True


In [11]:
#----------------------
#Load Raw Data & Strip Whitespaces
#-----------------------------

df = pd.read_csv(RAW_PATH, low_memory=False)

# Clean column headers and string padding
df.columns = df.columns.str.strip()
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].str.strip()

log.append(("rows_raw", len(df)))
print(f"Loaded Raw Shape: {df.shape}")
df.head(3)

Loaded Raw Shape: (79330, 31)


,IsCanceled,LeadTime,ArrivalDateYear,ArrivalDateMonth,ArrivalDateWeekNumber,ArrivalDateDayOfMonth,StaysInWeekendNights,StaysInWeekNights,Adults,Children,...,DepositType,Agent,Company,DaysInWaitingList,CustomerType,ADR,RequiredCarParkingSpaces,TotalOfSpecialRequests,ReservationStatus,ReservationStatusDate
0,0,6,2015,July,27,1,0,2,1,0.0,...,No Deposit,6,NULL,0,Transient,0.0,0,0,Check-Out,2015-07-03
1,1,88,2015,July,27,1,0,4,2,0.0,...,No Deposit,9,NULL,0,Transient,76.5,0,1,Canceled,2015-07-01
2,1,65,2015,July,27,1,0,4,1,0.0,...,No Deposit,9,NULL,0,Transient,68.0,0,1,Canceled,2015-04-30


In [12]:
print("--- Missing values before imputation ---")
print(df[["Children", "Country", "Agent", "Company"]].isna().sum())

# Children: 4 missing -> fill with 0
df["Children"] = df["Children"].fillna(0).astype(int)

# Country: 24 missing -> fill with 'Unknown'
df["Country"] = df["Country"].fillna("Unknown").astype(str)

# Agent & Company IDs -> clean string identifiers, NULLs become 'None'
df["Agent"] = _id_to_str(df["Agent"])
df["Company"] = _id_to_str(df["Company"])

print("\n--- After Imputation & Formatting ---")
print("Agent 'None' count :", (df["Agent"] == "None").sum())
print("Company 'None' count:", (df["Company"] == "None").sum())
print("Country 'Unknown' count:", (df["Country"] == "Unknown").sum())

--- Missing values before imputation ---
Children     4
Country     24
Agent        0
Company      0
dtype: int64

--- After Imputation & Formatting ---
Agent 'None' count : 8131
Company 'None' count: 75641
Country 'Unknown' count: 24


In [7]:
# Drop negative ADR or extreme typo (> 1000, e.g., the 5400.0 row)
adr_mask = (df["ADR"] < 0) | (df["ADR"] > ADR_MAX_VALID)
df = track_drop(df, adr_mask, "adr_invalid")

[*] adr_invalid           : Dropped     1 rows | Remaining: 79329


In [13]:
# Drop bookings with 0 total nights
zero_nights_mask = (df["StaysInWeekendNights"] + df["StaysInWeekNights"]) == 0
df = track_drop(df, zero_nights_mask, "zero_night_bookings")

[*] zero_night_bookings   : Dropped   331 rows | Remaining: 78999


In [14]:
# 1. Phantom bookings (0 adults, 0 children, 0 babies)
zero_guests_mask = (df["Adults"] + df["Children"] + df["Babies"]) == 0
df = track_drop(df, zero_guests_mask, "zero_guests")

# 2. Rogue kids-only bookings (Option C: 0 adults with kids, NOT Transient-Party and NOT Room B)
rogue_kids_mask = (
    (df["Adults"] == 0) & 
    ((df["Children"] > 0) | (df["Babies"] > 0)) & 
    (df["CustomerType"] != "Transient-Party") & 
    (df["ReservedRoomType"] != "B")
)
df = track_drop(df, rogue_kids_mask, "rogue_kids_only")

[*] zero_guests           : Dropped   100 rows | Remaining: 78899
[*] rogue_kids_only       : Dropped     8 rows | Remaining: 78891


In [16]:
# [7] Deduplication: Retain exactly 1 unique record per duplicate group

# 1. Total rows that belong to any duplicate pattern (for audit reporting)
all_dups_count = df.duplicated(keep=False).sum()

# 2. Extra duplicate occurrences (keeps 1 unique copy, flags the rest as True)
dup_mask = df.duplicated(keep="first")

print(f"Total rows in duplicate groups : {all_dups_count:>6}")
print(f"Extra duplicate copies to drop : {dup_mask.sum():>6}")
print(f"unique remaining : {all_dups_count - dup_mask.sum():>6}")

Total rows in duplicate groups :  31683
Extra duplicate copies to drop :  25856
unique remaining :   5827


In [17]:
# Apply drop using the tracking function
df = track_drop(df, dup_mask, "exact_duplicates")

# Reset index & record final row count
df = df.reset_index(drop=True)
log.append(("rows_after_cleaning", len(df)))

[*] exact_duplicates      : Dropped 25856 rows | Remaining: 53035


In [18]:
# 3. Sanity Check: Ensure zero duplicate rows remain
assert df.duplicated().sum() == 0, "Duplicate rows still exist!"
print(f"Verification: {df.duplicated().sum()} duplicates remain.")
print(f"Final clean dataset shape      : {df.shape}")

Verification: 0 duplicates remain.
Final clean dataset shape      : (53035, 31)


In [19]:
# Calendar derivations
df["arrival_month"] = df["ArrivalDateMonth"].map(MONTHS)
df["arrival_date"] = pd.to_datetime(dict(
    year=df["ArrivalDateYear"], 
    month=df["arrival_month"], 
    day=df["ArrivalDateDayOfMonth"]
))
df["arrival_dow"] = df["arrival_date"].dt.dayofweek

In [20]:
# Total guest and stay aggregations
df["total_nights"] = df["StaysInWeekendNights"] + df["StaysInWeekNights"]
df["total_guests"] = df["Adults"] + df["Children"] + df["Babies"]
df["has_children"] = ((df["Children"] + df["Babies"]) > 0).astype(int)

df[["arrival_date", "arrival_dow", "total_nights", "total_guests", "has_children"]].head(3)

,arrival_date,arrival_dow,total_nights,total_guests,has_children
0,2015-07-01,2,2,1,0
1,2015-07-01,2,4,2,0
2,2015-07-01,2,4,1,0


In [21]:
# Agent & Company indicator flags
df["has_agent"] = (df["Agent"] != "None").astype(int)
df["has_company"] = (df["Company"] != "None").astype(int)

In [22]:
# Guest loyalty history
df["has_previous_cancellations"] = (df["PreviousCancellations"] > 0).astype(int)
df["has_past_stays"] = (df["PreviousBookingsNotCanceled"] > 0).astype(int)
df["has_history"] = (((df["PreviousCancellations"] + df["PreviousBookingsNotCanceled"]) > 0) | (df["IsRepeatedGuest"] == 1)).astype(int)

In [23]:
# Booking latency & special features
df["has_booking_changes"] = (df["BookingChanges"] > 0).astype(int)
df["requires_parking"] = (df["RequiredCarParkingSpaces"] > 0).astype(int)
df["has_waiting_list"] = (df["DaysInWaitingList"] > 0).astype(int)
df["waiting_lead_ratio"] = df["DaysInWaitingList"] / (df["LeadTime"] + 1.0)
df["room_changed"] = (df["ReservedRoomType"] != df["AssignedRoomType"]).astype(int)

In [24]:
print(f"Total features after engineering: {df.shape[1]}")
df[["waiting_lead_ratio", "has_history", "requires_parking", "room_changed"]].head(3)

Total features after engineering: 47


,waiting_lead_ratio,has_history,requires_parking,room_changed
0,0.0,0,0,0
1,0.0,0,0,0
2,0.0,0,0,0


In [25]:
# Ensure output directories exist
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Save cleaned dataframe
df.to_csv(CLEAN_PATH, index=False)
print(f"Saved cleaned data ({df.shape[0]} rows, {df.shape[1]} cols) -> {CLEAN_PATH}")

# Save cleaning report log
df_report = pd.DataFrame(log, columns=["step", "rows"])
df_report.to_csv(REPORT_PATH, index=False)

print("\n--- Cleaning Report Summary ---")
print(df_report.to_string(index=False))

Saved cleaned data (53035 rows, 47 cols) -> d:\UNI\Fall 2026\CSE382- Intro. to ML\Project\City-Hotel-Booking\data\processed\H2_clean.csv

--- Cleaning Report Summary ---
               step  rows
           rows_raw 79330
zero_night_bookings   331
        zero_guests   100
    rogue_kids_only     8
   exact_duplicates 25856
rows_after_cleaning 53035
